# Week 2 Task – Data Visualization and Insight Communication using R
## Telco Customer Churn Analysis

**Dataset:** Telco Customer Churn  
**Language:** R  
**Libraries:** ggplot2, dplyr, readr, scales  
**Environment:** Google Colab

This notebook analyzes customer churn and communicates insights using multiple visualization techniques.

## 1. Install and Load Required Libraries
Run this cell first.

In [ ]:
install.packages(c("ggplot2", "dplyr", "readr", "scales"), repos="https://cloud.r-project.org")

library(ggplot2)
library(dplyr)
library(readr)
library(scales)

## 2. Upload the Dataset

In Google Colab, run the next cell and upload:

`WA_Fn-UseC_-Telco-Customer-Churn.csv`

If your file has a different name, change the filename in the next cell.

In [ ]:
# Check files available in the Colab working directory
list.files()

In [ ]:
# Read the uploaded CSV
file_name <- "WA_Fn-UseC_-Telco-Customer-Churn.csv"

telco <- read_csv(file_name, show_col_types = FALSE)

head(telco)
dim(telco)

## 3. Data Overview

In [ ]:
str(telco)
summary(telco)

# Number of missing values in each column
colSums(is.na(telco))

# Number of duplicate rows
sum(duplicated(telco))

## 4. Data Preparation

`TotalCharges` is stored as character/text in the original dataset because some records contain blank values. It is converted to numeric for analysis.

In [ ]:
telco <- telco %>%
  mutate(
    TotalCharges_num = as.numeric(TotalCharges),
    ChurnFlag = ifelse(Churn == "Yes", 1, 0)
  )

# Check the converted variable
summary(telco$TotalCharges_num)

# Overall churn rate
churn_rate <- mean(telco$ChurnFlag) * 100
churn_rate

# Visualization 1 – Customer Count by Contract Type

A bar chart is used to compare the number of customers in each contract category.

In [ ]:
p1 <- ggplot(telco, aes(x = Contract)) +
  geom_bar() +
  labs(
    title = "Customer Count by Contract Type",
    x = "Contract Type",
    y = "Number of Customers"
  ) +
  theme_minimal()

p1

### Insight
The chart shows the composition of the customer base by contract type. Month-to-month contracts represent a large customer group, making this category important for churn analysis.

# Visualization 2 – Churn Distribution by Contract Type

A proportional stacked bar chart compares churn percentages across contract types.

In [ ]:
p2 <- ggplot(telco, aes(x = Contract, fill = Churn)) +
  geom_bar(position = "fill") +
  scale_y_continuous(labels = percent_format()) +
  labs(
    title = "Churn Distribution by Contract Type",
    x = "Contract Type",
    y = "Percentage of Customers",
    fill = "Churn"
  ) +
  theme_minimal()

p2

### Insight
Month-to-month customers have a noticeably higher churn proportion than customers with one-year or two-year contracts. This suggests contract duration is an important factor associated with customer retention.

# Visualization 3 – Distribution of Monthly Charges

A histogram shows the spread and concentration of monthly customer charges.

In [ ]:
p3 <- ggplot(telco, aes(x = MonthlyCharges)) +
  geom_histogram(bins = 30) +
  labs(
    title = "Distribution of Monthly Charges",
    x = "Monthly Charges",
    y = "Frequency"
  ) +
  theme_minimal()

p3

### Insight
The histogram shows how monthly charges are distributed and helps identify the common billing range and unusually high or low charges.

# Visualization 4 – Monthly Charges by Churn Status

A box plot compares the distribution of monthly charges between customers who churned and customers who stayed.

In [ ]:
p4 <- ggplot(telco, aes(x = Churn, y = MonthlyCharges)) +
  geom_boxplot() +
  labs(
    title = "Monthly Charges by Churn Status",
    x = "Churn",
    y = "Monthly Charges"
  ) +
  theme_minimal()

p4

### Insight
The box plot helps compare the median and spread of monthly charges for churned and retained customers. Differences in these distributions can help identify customer groups that may need targeted retention efforts.

# Visualization 5 – Tenure vs Total Charges

A scatter plot is used to examine the relationship between customer tenure and accumulated total charges.

In [ ]:
p5 <- ggplot(
  telco %>% filter(!is.na(TotalCharges_num)),
  aes(x = tenure, y = TotalCharges_num, color = Churn)
) +
  geom_point(alpha = 0.4) +
  labs(
    title = "Tenure vs Total Charges",
    x = "Tenure (months)",
    y = "Total Charges",
    color = "Churn"
  ) +
  theme_minimal()

p5

### Insight
There is a strong positive relationship between tenure and total charges because customers who remain longer generally accumulate more charges. The churn grouping provides an additional view of customer behavior.

# Visualization 6 – Churn Distribution by Internet Service

In [ ]:
p6 <- ggplot(telco, aes(x = InternetService, fill = Churn)) +
  geom_bar(position = "fill") +
  scale_y_continuous(labels = percent_format()) +
  labs(
    title = "Churn Distribution by Internet Service",
    x = "Internet Service",
    y = "Percentage of Customers",
    fill = "Churn"
  ) +
  theme_minimal()

p6

### Insight
Churn proportions differ between internet service categories. This helps identify service groups where customer retention may require additional attention.

# Visualization 7 – Churn Rate by Customer Tenure

A line chart is appropriate because tenure is an ordered numerical variable.

In [ ]:
tenure_churn <- telco %>%
  group_by(tenure) %>%
  summarise(
    churn_rate = mean(ChurnFlag) * 100,
    customers = n(),
    .groups = "drop"
  )

p7 <- ggplot(tenure_churn, aes(x = tenure, y = churn_rate)) +
  geom_line() +
  labs(
    title = "Churn Rate by Customer Tenure",
    x = "Tenure (months)",
    y = "Churn Rate (%)"
  ) +
  theme_minimal()

p7

### Insight
The line chart helps identify how churn changes across customer tenure. Shorter-tenure customers generally show greater churn risk, while long-term customers tend to be more stable.

# Visualization 8 – Churn Rate by Payment Method

A horizontal bar chart is used because payment-method names can be relatively long.

In [ ]:
payment_churn <- telco %>%
  group_by(PaymentMethod) %>%
  summarise(
    churn_rate = mean(ChurnFlag) * 100,
    .groups = "drop"
  )

p8 <- ggplot(
  payment_churn,
  aes(x = reorder(PaymentMethod, churn_rate), y = churn_rate)
) +
  geom_col() +
  coord_flip() +
  labs(
    title = "Churn Rate by Payment Method",
    x = "Payment Method",
    y = "Churn Rate (%)"
  ) +
  theme_minimal()

p8

### Insight
The visualization highlights differences in churn rates across payment methods. Such differences can help the business investigate whether payment experience or customer profile is associated with retention.

# 5. Additional Summary Statistics

In [ ]:
cat("Total customers:", nrow(telco), "\n")
cat("Overall churn rate:", round(churn_rate, 2), "%\n")
cat("Average monthly charges:", round(mean(telco$MonthlyCharges, na.rm = TRUE), 2), "\n")
cat("Median tenure:", median(telco$tenure, na.rm = TRUE), "months\n")

# Churn count
telco %>%
  count(Churn) %>%
  mutate(Percentage = n / sum(n) * 100)

# 6. Export Graphs as High-Quality Images

The following code saves all visualizations as PNG files for inserting into the final DOC report.

In [ ]:
dir.create("plots", showWarnings = FALSE)

ggsave("plots/01_contract_bar.png", p1, width = 8, height = 5, dpi = 300)
ggsave("plots/02_contract_churn.png", p2, width = 8, height = 5, dpi = 300)
ggsave("plots/03_monthly_histogram.png", p3, width = 8, height = 5, dpi = 300)
ggsave("plots/04_monthly_boxplot.png", p4, width = 8, height = 5, dpi = 300)
ggsave("plots/05_tenure_total_scatter.png", p5, width = 8, height = 5, dpi = 300)
ggsave("plots/06_internet_churn.png", p6, width = 8, height = 5, dpi = 300)
ggsave("plots/07_tenure_line.png", p7, width = 8, height = 5, dpi = 300)
ggsave("plots/08_payment_churn.png", p8, width = 8, height = 5, dpi = 300)

list.files("plots")

# 7. Overall Findings

- Contract type is strongly associated with churn behavior.
- Month-to-month customers show higher churn than customers on longer contracts.
- Customer tenure and total charges have a positive relationship.
- Short-tenure customers are an important group for retention analysis.
- Monthly charge distributions differ between churned and retained customers.
- Internet service and payment method show useful differences in churn rates.

## Conclusion

R and ggplot2 provide an effective way to communicate customer churn patterns through clear visualizations. The combination of bar charts, histograms, box plots, scatter plots, and line charts helps a non-technical audience understand customer behavior and identify potential retention opportunities.